## Loading the Dataset

In [2]:
import pandas as pd
recipe2M = pd.read_csv('recipes_data.csv')

In [3]:
recipe2M.head()

,title,ingredients,directions,link,source,NER,site
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""bite size shredded rice biscuits"", ""vanilla""...",www.cookbooks.com
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""cream of mushroom soup"", ""beef"", ""sour cream...",www.cookbooks.com
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""pepper"", ""cream cheese"", ""gar...",www.cookbooks.com
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken gravy"", ""cream of mushroom soup"", ""c...",www.cookbooks.com
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""graham cracker crumbs"", ""powdered sugar"", ""p...",www.cookbooks.com


## Removing unnecessary columns & nulls

In [4]:
recipe2M_cleaned=recipe2M.drop(columns=['link', 'source', 'site'], inplace=False)
recipe2M_cleaned.dropna()

,title,ingredients,directions,NER
0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...","[""bite size shredded rice biscuits"", ""vanilla""..."
1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....","[""cream of mushroom soup"", ""beef"", ""sour cream..."
2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...","[""frozen corn"", ""pepper"", ""cream cheese"", ""gar..."
3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...","[""chicken gravy"", ""cream of mushroom soup"", ""c..."
4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...","[""graham cracker crumbs"", ""powdered sugar"", ""p..."
...,...,...,...,...
2231137,Sunny's Fake Crepes,"[""1/2 cup chocolate hazelnut spread (recommend...","[""Spread hazelnut spread on 1 side of each tor...","[""chocolate hazelnut spread"", ""marshmallows"", ..."
2231138,Devil Eggs,"[""1 dozen eggs"", ""1 paprika"", ""1 salt and pepp...","[""Boil eggs on medium for 30mins."", ""Then cool...","[""choice"", ""miracle whip"", ""eggs"", ""relish"", ""..."
2231139,Extremely Easy and Quick - Namul Daikon Salad,"[""150 grams Daikon radish"", ""1 tbsp Sesame oil...","[""Julienne the daikon and squeeze out the exce...","[""soy sauce"", ""radish"", ""white sesame seeds"", ..."
2231140,Pan-Roasted Pork Chops With Apple Fritters,"[""1 cup apple cider"", ""6 tablespoons sugar"", ""...","[""In a large bowl, mix the apple cider with 4 ...","[""apple cider"", ""egg"", ""sugar"", ""freshly groun..."


## Removing recipes with directions contatining the word "step"

In [5]:

recipe2M_cleaned = recipe2M_cleaned[~recipe2M_cleaned['directions'].str.contains('step', case=False, na=False)]
recipe2M_cleaned['title'].count()


2206617

## Removing recipes with at most 1 ingredient

In [6]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['ingredients'].apply(lambda x: len([i for i in x if i.strip()]) <= 1)].index, inplace=True)
recipe2M_cleaned['title'].count()


2206617

## Removing recipes with instructions less than 10 characters

In [7]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['directions'].apply(lambda x: not all(len(i.strip()) < 10 for i in x if i.strip()))].index, inplace=True)
recipe2M_cleaned['title'].count()

2206617

## Removing recipes with title less than 4 characters 

In [8]:
recipe2M_cleaned.drop(recipe2M_cleaned[recipe2M_cleaned['title'].apply(lambda x: len(str(x)) < 4 if pd.notnull(x) else False)].index, inplace=True)
recipe2M_cleaned['title'].count()

2206372

In [9]:
import json
def is_empty_list(column_data):
    try:
        # Parse the JSON string into a Python list
        parsed_data = json.loads(column_data)
        return isinstance(parsed_data, list) and not parsed_data
    except json.JSONDecodeError:
        # If json.loads fails, consider it invalid and drop
        return True

# Apply the function to identify rows to drop
columns_to_check = ['NER', 'ingredients', 'directions']  # Columns to validate
for column in columns_to_check:
    recipe2M_cleaned = recipe2M_cleaned[~recipe2M_cleaned[column].apply(is_empty_list)]

## Splitting Data For Test And Train

In [10]:
from sklearn.model_selection import train_test_split

train_recipes,test_recipes = train_test_split(recipe2M_cleaned,test_size=0.1,random_state=42)

test_recipes.head()


,title,ingredients,directions,NER
585051,Tuna Casserole,"[""1 1/2 lb. noodles"", ""1 large can tuna"", ""1 c...","[""Cook noodles in salt water until soft."", ""Ad...","[""cream of chicken soup"", ""noodles"", ""tuna"", ""..."
1651930,Black Bean And Mango Salsa Recipe,"[""1 can (15 ounce) black beans, rinsed and liq...","[""Chop and mix all ingredients."", ""Refrigerate...","[""black beans"", ""tomato"", ""pepper"", ""salt"", ""o..."
726042,Toasted Butter Pecan Cake,"[""2 c. pecans, chopped"", ""1 1/4 c. butter"", ""3...","[""Toast pecans in 1/4 cup butter in oven for 2...","[""sugar"", ""vanilla"", ""double-acting baking pow..."
538859,Vegetable Soup,"[""1 lb. hamburger meat"", ""1 onion, chopped"", ""...","[""Salt and pepper to taste."", ""Water as needed...","[""cabbage"", ""corn"", ""tomatoes"", ""onion"", ""butt..."
1801484,Bentzi's One Meal Meatloaf,"[""3 lbs ground beef"", ""1 (16 ounce) bag frozen...","[""Combine everything except ketchup and mushro...","[""tomatoes"", ""pepper"", ""eggs"", ""chili powder"",..."


## Adding Control Tags

In [22]:
# Function to format a single recipe
def format_recipe(row,test=False):
    formatted_recipe = []
    
    # Add RECIPE_START token
    formatted_recipe.append("<RECIPE_START>")
    
    # Add Inputs (ner column)
    
    inputs = json.loads(row['NER']) # Split NER data into a list
    formatted_recipe.append(f"<INPUT_START> {inputs[0].strip()}")
    for i in range(1,len(inputs)):
        formatted_recipe.append(f"<NEXT_INPUT> {inputs[i].strip()}")
    formatted_recipe.append("<INPUT_END>")
    if test:
        return ' '.join(formatted_recipe)
    # Add Ingredients (ingredients column)
    
    ingredients = json.loads(row['ingredients'])  # Split ingredients into a list
    formatted_recipe.append(f"<INGR_START> {ingredients[0].strip()}")
    for i in range(1,len(ingredients)):
        formatted_recipe.append(f"<NEXT_INGR> {ingredients[i].strip()}")
    formatted_recipe.append("<INGR_END>")
    
    # Add Instructions (directions column)
    
    instructions = json.loads(row['directions'])  # Split directions into steps
    formatted_recipe.append(f"<INSTR_START> {instructions[0].strip()}")
    for i in range(1,len(instructions)):
        formatted_recipe.append(f"<NEXT_INSTR> {instructions[i].strip()}.")
    formatted_recipe.append("<INSTR_END>")
    
    # Add Title (title column)
    formatted_recipe.append(f"<TITLE_START> {row['title']} <TITLE_END>")
    
    # Add RECIPE_END token
    formatted_recipe.append("<RECIPE_END>")
    
    return ' '.join(formatted_recipe)

In [23]:
# Apply formatting to the entire dataset
train_recipes['formatted_recipe'] = train_recipes.apply(format_recipe, axis=1)
test_recipes['formatted_recipe'] = test_recipes.apply(lambda x: format_recipe(x, True), axis=1)

In [24]:
train_recipes['formatted_recipe'][0]

'<RECIPE_START> <INPUT_START> bite size shredded rice biscuits <NEXT_INPUT> vanilla <NEXT_INPUT> brown sugar <NEXT_INPUT> nuts <NEXT_INPUT> milk <NEXT_INPUT> butter <INPUT_END> <INGR_START> 1 c. firmly packed brown sugar <NEXT_INGR> 1/2 c. evaporated milk <NEXT_INGR> 1/2 tsp. vanilla <NEXT_INGR> 1/2 c. broken nuts (pecans) <NEXT_INGR> 2 Tbsp. butter or margarine <NEXT_INGR> 3 1/2 c. bite size shredded rice biscuits <INGR_END> <INSTR_START> In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine. <NEXT_INSTR> Stir over medium heat until mixture bubbles all over top.. <NEXT_INSTR> Boil and stir 5 minutes more. Take off heat.. <NEXT_INSTR> Stir in vanilla and cereal; mix well.. <NEXT_INSTR> Using 2 teaspoons, drop and shape into 30 clusters on wax paper.. <NEXT_INSTR> Let stand until firm, about 30 minutes.. <INSTR_END> <TITLE_START> No-Bake Nut Cookies <TITLE_END> <RECIPE_END>'

In [28]:
test_recipes['formatted_recipe'][585051]

'<RECIPE_START> <INPUT_START> cream of chicken soup <NEXT_INPUT> noodles <NEXT_INPUT> tuna <NEXT_INPUT> cheddar cheese <NEXT_INPUT> onion <NEXT_INPUT> pimientos <NEXT_INPUT> green pepper <INPUT_END>'

In [29]:
test_data=test_recipes['formatted_recipe']
train_data=train_recipes['formatted_recipe']

In [31]:
test_data.to_csv("test_data.csv",index=False)

train_data.to_csv("train_data.csv",index=False)